<a href="https://colab.research.google.com/github/mizinco/sam2-recognition-limit-observer/blob/main/SAM2_observation_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Tue Jun 16 08:59:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd /content/sam2
!pip install -e . -q
!pip install opencv-python matplotlib -q

/content
Cloning into 'sam2'...
remote: Enumerating objects: 1107, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 1107 (delta 5), reused 2 (delta 2), pack-reused 1098 (from 3)
Receiving objects: 100% (1107/1107), 134.92 MiB | 24.03 MiB/s, done.
Resolving deltas: 100% (381/381), done.
/content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.9 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done


In [ ]:
import os
ckpt = '/content/sam2/checkpoints/sam2.1_hiera_large.pt'
if not os.path.exists(ckpt):
    !wget -q -P /content/sam2/checkpoints/ \
        https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
print(f'{ckpt}: {os.path.getsize(ckpt)/1e6:.0f} MB')

/content/sam2/checkpoints/sam2.1_hiera_large.pt: 898 MB


In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import cv2

sys.path.insert(0, '/content/sam2')
from sam2.build_sam import build_sam2_video_predictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.autocast(device_type=device.type, dtype=torch.bfloat16).__enter__()
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print('using device:', device)

using device: cuda


In [ ]:
sam2_checkpoint = '/content/sam2/checkpoints/sam2.1_hiera_large.pt'
model_cfg = 'configs/sam2.1/sam2.1_hiera_l.yaml'
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)
print('predictor ready')

predictor ready


In [ ]:
import os
import json
import subprocess
import tempfile

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
import cv2
from PIL import Image
import gradio as gr


# ============================================================================
# 指標関数（マスクの状態を数値化する関数群）
# ============================================================================

# 前のフレームからのマスク面積の変化率を計算
def mask_area_change_rate(masks):
    areas = masks.sum(axis=(1, 2)).astype(np.float64)
    change_rate = np.zeros(len(areas), dtype=np.float32)
    for t in range(1, len(areas)):
        prev, curr = areas[t - 1], areas[t]
        if prev == 0:
            change_rate[t] = 0.0 if curr == 0 else float('inf')
        else:
            change_rate[t] = (curr - prev) / prev
    return change_rate

# マスク内の孤立した領域（連結成分）の数をカウント（断片化の指標）
def connected_components_count(masks, min_size=50):
    counts = np.zeros(len(masks), dtype=np.int32)
    for t in range(len(masks)):
        if masks[t].sum() == 0:
            counts[t] = 0
            continue
        labeled, n = ndimage.label(masks[t])
        if n == 0:
            counts[t] = 0
        elif min_size > 0:
            # 指定サイズ以下の小さなノイズを除去してカウント
            sizes = ndimage.sum(masks[t], labeled, range(1, n + 1))
            counts[t] = int((sizes >= min_size).sum())
        else:
            counts[t] = n
    return counts

# 全マスク面積に対する「最大の破片」の割合を計算（形状のまとまり具合）
def largest_component_ratio(masks):
    ratios = np.ones(len(masks), dtype=np.float32)
    for t in range(len(masks)):
        total = masks[t].sum()
        if total == 0:
            ratios[t] = 1.0
            continue
        labeled, n = ndimage.label(masks[t])
        if n == 0:
            ratios[t] = 1.0
            continue
        sizes = ndimage.sum(masks[t], labeled, range(1, n + 1))
        ratios[t] = float(sizes.max() / total)
    return ratios

# 重心を合わせた状態での時間的な形状一致度(IoU)を計算（変形の激しさを測定）
def shape_iou_temporal(masks):
    T = len(masks)
    iou = np.ones(T, dtype=np.float32)
    if T == 0:
        return iou
    H, W = masks[0].shape

    def _centroid(m):
        ys, xs = np.where(m)
        if len(ys) == 0:
            return None
        return float(ys.mean()), float(xs.mean())

    for t in range(1, T):
        m_prev, m_curr = masks[t - 1], masks[t]
        if m_prev.sum() == 0 or m_curr.sum() == 0:
            iou[t] = 1.0
            continue
        # 重心を計算して位置ズレを補正し、純粋な「形」の差を見る
        c_prev = _centroid(m_prev)
        c_curr = _centroid(m_curr)
        dy = int(round(c_prev[0] - c_curr[0]))
        dx = int(round(c_prev[1] - c_curr[1]))
        shifted = np.zeros_like(m_curr)
        ys, ye = max(0, dy), min(H, H + dy)
        xs, xe = max(0, dx), min(W, W + dx)
        if ys < ye and xs < xe:
            shifted[ys:ye, xs:xe] = m_curr[ys - dy:ye - dy, xs - dx:xe - dx]
        inter = np.logical_and(m_prev, shifted).sum()
        union = np.logical_or(m_prev, shifted).sum()
        iou[t] = float(inter / union) if union > 0 else 1.0
    return iou

# マスクの消失判定（閾値以下になったフレームを特定）
def mask_disappearance(masks, threshold_abs=50, threshold_rel=0.01, ref_frames=5):
    areas = masks.sum(axis=(1, 2)).astype(np.float64)
    ref = areas[:max(ref_frames, 1)]
    ref_area = float(ref.mean()) if len(ref) > 0 else 1.0
    # 絶対閾値と相対閾値の大きい方を採用
    threshold = max(float(threshold_abs), ref_area * threshold_rel)
    is_dis = areas < threshold
    cons = np.zeros(len(areas), dtype=np.int32)
    for t in range(len(areas)):
        if is_dis[t]:
            cons[t] = (cons[t - 1] if t > 0 else 0) + 1
        else:
            cons[t] = 0
    return {
        'areas': areas,
        'is_disappeared': is_dis,
        'consecutive': cons,
        'threshold': threshold,
        'reference_area': ref_area,
    }

# 全ての指標を一括計算して辞書形式で返す
def compute_all_metrics(masks, threshold_abs=50, min_size=50):
    area_change = mask_area_change_rate(masks)
    components = connected_components_count(masks, min_size=min_size)
    largest = largest_component_ratio(masks)
    disappear = mask_disappearance(masks, threshold_abs=threshold_abs)
    shape_iou = shape_iou_temporal(masks)
    return {
        'frame_count': int(len(masks)),
        'areas': disappear['areas'].astype(float).tolist(),
        'area_change_rate': area_change.tolist(),
        'connected_components': components.tolist(),
        'largest_component_ratio': largest.tolist(),
        'shape_iou_temporal': shape_iou.tolist(),
        'is_disappeared': disappear['is_disappeared'].tolist(),
        'consecutive_disappearance': disappear['consecutive'].tolist(),
        'reference_area': float(disappear['reference_area']),
        'disappearance_threshold': float(disappear['threshold']),
        'min_component_size': int(min_size),
    }


# ============================================================================
# 可視化パネルの作成
# ============================================================================

def make_5panel_plot(metrics, case_name='session'):
    # 5つの指標を並べたグラフを生成
    plt.rcParams['font.family'] = 'DejaVu Sans'
    fig, axes = plt.subplots(5, 1, figsize=(11, 12), sharex=True)
    frames = np.arange(metrics['frame_count'])

    # 1. 面積推移と消失閾値
    axes[0].plot(frames, metrics['areas'], color='steelblue', linewidth=1.5)
    axes[0].axhline(
        metrics['disappearance_threshold'], color='red', linestyle='--',
        label=f"Disappearance threshold ({metrics['disappearance_threshold']:.0f}px)",
    )
    axes[0].set_ylabel('Mask area (px)')
    axes[0].set_title(f'{case_name} - mask area over time')
    axes[0].legend(loc='upper right')
    axes[0].grid(alpha=0.3)

    # 2. 面積変化率（急激な変化を検知）
    rates = np.array(metrics['area_change_rate'], dtype=float)
    rates_disp = np.clip(rates, -2.0, 2.0)
    axes[1].plot(frames, rates_disp, color='darkorange', linewidth=1.5)
    axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='+-50%')
    axes[1].axhline(-0.5, color='red', linestyle='--', alpha=0.5)
    axes[1].set_ylabel('Area change rate')
    axes[1].set_title('Indicator 1: Mask area change rate (clipped to +-2.0)')
    axes[1].legend(loc='upper right')
    axes[1].grid(alpha=0.3)

    # 3. 連結成分数（断片化・ノイズの混入）
    axes[2].plot(
        frames, metrics['connected_components'],
        color='seagreen', linewidth=1.5, marker='o', markersize=3,
    )
    axes[2].set_ylabel('Component count')
    axes[2].set_title(
        f'Indicator 2: Connected components (>={metrics["min_component_size"]}px only)'
    )
    axes[2].grid(alpha=0.3)

    # 4. 消失状態（赤いエリアは消失判定中）
    is_dis = np.array(metrics['is_disappeared'])
    cons = np.array(metrics['consecutive_disappearance'])
    axes[3].fill_between(
        frames, 0, is_dis.astype(int),
        alpha=0.4, color='crimson', step='post', label='Disappeared',
    )
    if cons.max() > 0:
        axes[3].plot(
            frames, cons / max(cons.max(), 1),
            color='crimson', linewidth=1.5, label='Consecutive (normalized)',
        )
    axes[3].set_ylim(-0.05, 1.1)
    axes[3].set_ylabel('Disappearance')
    axes[3].set_title(
        f'Indicator 3: Mask disappearance (max consecutive: {int(cons.max())} frames)'
    )
    axes[3].legend(loc='upper left')
    axes[3].grid(alpha=0.3)

    # 5. 形状の安定性（IoUが低い＝形が崩壊している）
    siou = np.array(metrics['shape_iou_temporal'], dtype=float)
    axes[4].plot(frames, siou, color='purple', linewidth=1.5)
    axes[4].axhline(0.8, color='red', linestyle='--', alpha=0.5, label='IoU 0.8')
    axes[4].set_ylim(0, 1.05)
    min_idx = int(np.argmin(siou))
    axes[4].set_title(
        f'Indicator 4: Shape stability (centroid-aligned IoU, min '
        f'{float(siou.min()):.3f} at frame {min_idx})'
    )
    axes[4].set_ylabel('Shape IoU')
    axes[4].set_xlabel('Frame')
    axes[4].legend(loc='lower right')
    axes[4].grid(alpha=0.3)

    plt.tight_layout()
    return fig


# ============================================================================
# フォーカス動画の生成（追従クロップ処理）
# ============================================================================

def make_focus_video(
    state,
    output_size: int = 400,
    constant_ratio: bool = True,
    target_mask_ratio: float = 0.3,
    iou_warning_threshold: float = 0.5,
    show_mask_overlay: bool = True,
    progress=gr.Progress(),
):
    # エラーチェック
    if state is None or state.get('masks') is None:
        return None, None, 'Run tracking first to generate focus video'

    SMOOTHING_ALPHA = 0.15      # ズーム倍率の滑らかさ（EMA係数）
    MIN_WINDOW_SIZE = 100       # 寄りすぎ防止の最小窓サイズ
    output_size = int(output_size)
    target_mask_ratio = float(target_mask_ratio)
    iou_warning_threshold = float(iou_warning_threshold)

    masks = state['masks']
    frame_dir = state['frame_dir']
    out_dir = state['out_dir']

    # 指標の読み込み（品質警告ラベル用）
    metrics_json_path = state.get('metrics_json')
    if metrics_json_path and os.path.exists(metrics_json_path):
        with open(metrics_json_path) as f:
            metrics = json.load(f)
        siou = np.array(metrics.get('shape_iou_temporal', []), dtype=float)
        is_dis = np.array(metrics.get('is_disappeared', []), dtype=bool)
    else:
        siou = np.ones(len(masks), dtype=float)
        is_dis = np.zeros(len(masks), dtype=bool)

    frames_list = sorted(os.listdir(frame_dir))
    H_full, W_full = state['first_frame'].shape[:2]
    max_window = min(H_full, W_full)

    focus_dir = os.path.join(out_dir, 'focus')
    os.makedirs(focus_dir, exist_ok=True)

    # --- 1. ウィンドウサイズの計算と平滑化 ---
    raw_window_sizes = []
    last_ws = output_size
    for i in range(len(masks)):
        m = masks[i]
        if m.any():
            mask_area = float(m.sum())
            if constant_ratio:
                # 面積から「画面内の比率が一定になるサイズ」を逆算
                ws = int(round(np.sqrt(mask_area / max(target_mask_ratio, 0.01))))
            else:
                ws = output_size
            ws = max(MIN_WINDOW_SIZE, min(max_window, ws))
            last_ws = ws
        else:
            ws = last_ws  # 消失時は直前のサイズを維持
        raw_window_sizes.append(ws)

    # 指数移動平均(EMA)でズームのガタつきを抑える
    smoothed_sizes = []
    ema = float(raw_window_sizes[0]) if raw_window_sizes else float(output_size)
    for ws in raw_window_sizes:
        ema = SMOOTHING_ALPHA * float(ws) + (1.0 - SMOOTHING_ALPHA) * ema
        smoothed_sizes.append(int(round(ema)))

    # --- 2. クロップ・描画処理 ---
    last_cy, last_cx = H_full // 2, W_full // 2

    progress(0, desc='Generating focus video')
    for i, fname in enumerate(frames_list):
        img = cv2.imread(os.path.join(frame_dir, fname))
        if img is None: continue

        # 重心計算（追従ターゲットの位置）
        m = masks[i] if i < len(masks) else None
        if m is not None and m.any():
            ys, xs = np.where(m)
            cy, cx = int(ys.mean()), int(xs.mean())
            last_cy, last_cx = cy, cx
        else:
            cy, cx = last_cy, last_cx

        ws = smoothed_sizes[i] if i < len(smoothed_sizes) else output_size
        half_h = ws // 2
        half_w = ws // 2

        # 切り出し範囲の決定
        y0, y1 = max(0, cy - half_h), min(H_full, cy + half_h)
        x0, x1 = max(0, cx - half_w), min(W_full, cx + half_w)
        crop = img[y0:y1, x0:x1].copy()

        # マスクのオーバーレイ表示（緑色の塗り）
        if show_mask_overlay and m is not None:
            mask_crop = m[y0:y1, x0:x1]
            if mask_crop.any():
                ov = crop.copy()
                ov[mask_crop] = (0, 255, 0)
                crop = cv2.addWeighted(crop, 0.6, ov, 0.4, 0)

        # 正方形にパディング補正
        actual_h, actual_w = crop.shape[:2]
        if actual_h < ws or actual_w < ws:
            padded = np.zeros((ws, ws, 3), dtype=np.uint8)
            pad_y, pad_x = (ws - actual_h) // 2, (ws - actual_w) // 2
            padded[pad_y:pad_y + actual_h, pad_x:pad_x + actual_w] = crop
            crop = padded

        # 指定サイズにリサイズ（ここで「動的ズーム」が完成する）
        if crop.shape[0] != output_size or crop.shape[1] != output_size:
            crop = cv2.resize(crop, (output_size, output_size), interpolation=cv2.INTER_LINEAR)

        # 品質警告（赤枠とラベルの焼き込み）
        warn = (i < len(siou) and float(siou[i]) < iou_warning_threshold) or (i < len(is_dis) and bool(is_dis[i]))
        if warn:
            cv2.rectangle(crop, (0, 0), (output_size - 1, output_size - 1), (0, 0, 255), 8)
            cv2.putText(crop, 'LOW QUALITY', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # 情報を書き込んで保存
        cv2.putText(crop, f'f={i} ws={ws}', (10, output_size - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.imwrite(os.path.join(focus_dir, f'{i:05d}.jpg'), crop)
        progress((i + 1) / len(frames_list), desc=f'Focus frame {i + 1}/{len(frames_list)}')

    # 動画ファイルを結合(ffmpeg)
    focus_mp4 = os.path.join(out_dir, 'focus_video.mp4')
    fps = state.get('target_fps', 10)
    subprocess.run(['ffmpeg', '-y', '-framerate', str(fps), '-i', os.path.join(focus_dir, '%05d.jpg'),
                    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', focus_mp4], check=True, capture_output=True)

    state['focus_mp4'] = focus_mp4
    return focus_mp4, focus_mp4, 'Focus video generated.'


# ============================================================================
# Gradio UI イベント処理
# ============================================================================

# 動画読み込みとフレーム抽出
def upload_video(video_path, target_fps, state):
    if video_path is None: return None, state, 'Upload a video first'
    out_dir = tempfile.mkdtemp(prefix='gradio_frames_')
    target_fps = int(target_fps)
    # ffmpegでリサイズとfps調整を行いつつ抽出
    vf = f'fps={target_fps},pad=ceil(iw/2)*2:ceil(ih/2)*2'
    subprocess.run(['ffmpeg', '-y', '-i', video_path, '-vf', vf, '-q:v', '2', os.path.join(out_dir, '%05d.jpg')], check=True)

    frames = sorted(os.listdir(out_dir))
    first = np.array(Image.open(os.path.join(out_dir, frames[0])))
    inference_state = predictor.init_state(video_path=out_dir)

    # 初期状態の辞書を作成
    state = {
        'frame_dir': out_dir, 'video_path': video_path, 'n_frames': len(frames),
        'target_fps': target_fps, 'first_frame': first, 'click_xy': None,
        'masks': None, 'out_dir': None, 'inference_state': inference_state,
    }
    return first, state, f'Loaded {len(frames)} frames. Please click on the target.'

# 追跡実行
def run_tracking(state, thr_abs, min_comp, progress=gr.Progress()):
    if state is None or state.get('click_xy') is None: return None, None, None, None, None, state, 'Click a point first'

    inference_state = state['inference_state']
    predictor.reset_state(inference_state)
    x, y = state['click_xy']

    # クリック地点を初期プロンプトとして追加
    predictor.add_new_points_or_box(inference_state=inference_state, frame_idx=0, obj_id=1,
                                    points=np.array([[x, y]], dtype=np.float32), labels=np.array([1], dtype=np.int32))

    frame_dir = state['frame_dir']
    frames_list = sorted(os.listdir(frame_dir))
    H, W = state['first_frame'].shape[:2]
    masks_all = np.zeros((len(frames_list), H, W), dtype=bool)

    # 全フレームの伝播推論を開始
    progress(0, desc='Propagating')
    for fidx, _obj_ids, mask_logits in predictor.propagate_in_video(inference_state):
        if fidx < len(frames_list):
            m = (mask_logits[0] > 0.0).cpu().numpy()[0]
            masks_all[fidx] = m
        progress((fidx + 1) / len(frames_list), desc=f'Frame {fidx + 1}/{len(frames_list)}')

    # 結果の保存とオーバーレイ動画作成
    out_dir = tempfile.mkdtemp(prefix='gradio_out_')
    overlay_dir = os.path.join(out_dir, 'overlay')
    os.makedirs(overlay_dir, exist_ok=True)
    for i, fname in enumerate(frames_list):
        img = cv2.imread(os.path.join(frame_dir, fname))
        if masks_all[i].any():
            overlay = img.copy(); overlay[masks_all[i]] = (0, 255, 0)
            img = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)
        cv2.imwrite(os.path.join(overlay_dir, f'{i:05d}.jpg'), img)

    tracked_mp4 = os.path.join(out_dir, 'tracked.mp4')
    subprocess.run(['ffmpeg', '-y', '-framerate', str(state['target_fps']), '-i', os.path.join(overlay_dir, '%05d.jpg'),
                    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', tracked_mp4], check=True)

    # 指標の計算
    metrics = compute_all_metrics(masks_all, threshold_abs=int(thr_abs), min_size=int(min_comp))
    metrics_json = os.path.join(out_dir, 'metrics.json')
    with open(metrics_json, 'w') as f: json.dump(metrics, f, indent=2)
    fig = make_5panel_plot(metrics)
    metrics_png = os.path.join(out_dir, 'metrics.png')
    fig.savefig(metrics_png, dpi=120, bbox_inches='tight')

    # 状態の更新
    state.update({'masks': masks_all, 'tracked_mp4': tracked_mp4, 'metrics_json': metrics_json, 'metrics_png': metrics_png, 'out_dir': out_dir})
    return tracked_mp4, fig, None, metrics_json, metrics_png, state, 'Tracking done.'

# 指標の再計算（追跡済みのマスクデータを使用）
def recompute_metrics(state, thr_abs, min_comp):
    if state is None or state.get('masks') is None: return None, None, None, 'Run tracking first'
    metrics = compute_all_metrics(state['masks'], threshold_abs=int(thr_abs), min_size=int(min_comp))
    fig = make_5panel_plot(metrics)
    metrics_png = os.path.join(state['out_dir'], 'metrics.png')
    fig.savefig(metrics_png, dpi=120, bbox_inches='tight')
    return fig, state['metrics_json'], metrics_png, 'Recomputed.'


# ============================================================================
# Gradio UI 定義
# ============================================================================

with gr.Blocks(title='SAM2 Recognition Limit Observer') as demo:
    gr.Markdown('# SAM2 Recognition Limit Observer')
    state = gr.State(value=None)

    with gr.Row():
        with gr.Column():
            video_in = gr.Video(label='Input video')
            fps_slider = gr.Slider(5, 30, value=10, label='Extract FPS')
            upload_btn = gr.Button('1. Load video')
            first_frame = gr.Image(label='2. Click on the target', interactive=False)
        with gr.Column():
            thr_abs = gr.Slider(0, 1000, value=50, label='Disappearance threshold (px)')
            min_comp = gr.Slider(0, 500, value=50, label='Min component size (px)')
            track_btn = gr.Button('3. Run tracking', variant='primary')
            recompute_btn = gr.Button('4. Recompute metrics')
            tracked_video = gr.Video(label='Tracked output')

    metrics_plot = gr.Plot(label='Analysis Metrics')

    with gr.Row():
        with gr.Column():
            gr.Markdown('### Focus Video Settings')
            out_sz = gr.Slider(200, 800, value=400, label='Output size')
            const_ratio = gr.Checkbox(value=True, label='Constant mask ratio')
            target_ratio = gr.Slider(0.05, 0.6, value=0.3, label='Target ratio')
            iou_thr = gr.Slider(0.0, 1.0, value=0.5, label='Quality warning threshold')
            focus_btn = gr.Button('5. Generate focus video', variant='primary')
        with gr.Column():
            focus_video_display = gr.Video(label='Focus video')

    # ── イベント結線 ──
    upload_btn.click(upload_video, [video_in, fps_slider, state], [first_frame, state])
    first_frame.select(lambda s, e: (s['first_frame'], {**s, 'click_xy': (int(e.index[0]), int(e.index[1]))}, 'Clicked'), [state], [first_frame, state])
    track_btn.click(run_tracking, [state, thr_abs, min_comp], [tracked_video, metrics_plot, gr.File(), gr.File(), gr.File(), state])
    recompute_btn.click(recompute_metrics, [state, thr_abs, min_comp], [metrics_plot, gr.File(), gr.File()])
    focus_btn.click(make_focus_video, [state, out_sz, const_ratio, target_ratio, iou_thr], [focus_video_display])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bc136f8808bc1fde1c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
